<a href="https://colab.research.google.com/github/harman1223-ai/csci-164/blob/main/solving_search_164AI.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

 sliding puzzle

In [ ]:
import heapq
import random
from collections import deque

class SlidingPuzzle:
    def __init__(self, size):
        self.size = size
        self.goal = tuple(range(1, size * size)) + (0,)

    def get_neighbors(self, state):
        neighbors = []
        idx = state.index(0)
        row, col = divmod(idx, self.size)
        directions = [(-1, 0, 'U'), (1, 0, 'D'), (0, -1, 'L'), (0, 1, 'R')]
        for dr, dc, move in directions:
            nr, nc = row + dr, col + dc
            if 0 <= nr < self.size and 0 <= nc < self.size:
                nidx = nr * self.size + nc
                new_state = list(state)
                new_state[idx], new_state[nidx] = new_state[nidx], new_state[idx]
                neighbors.append((tuple(new_state), move))
        return neighbors

    def manhattan(self, state):
        distance = 0
        for i, tile in enumerate(state):
            if tile == 0:
                continue
            goal_idx = tile - 1
            r1, c1 = divmod(i, self.size)
            r2, c2 = divmod(goal_idx, self.size)
            distance += abs(r1 - r2) + abs(c1 - c2)
        return distance

    def misplaced(self, state):
        return sum(1 for i, tile in enumerate(state) if tile != 0 and tile != self.goal[i])

    def random_walk(self, steps):
        state = self.goal
        last_move = None
        move_map = {'U': 'D', 'D': 'U', 'L': 'R', 'R': 'L'}
        sequence = []
        for _ in range(steps):
            neighbors = self.get_neighbors(state)
            valid = [(s, m) for s, m in neighbors if m != move_map.get(last_move)]
            state, last_move = random.choice(valid)
            sequence.append(last_move)
        return state, sequence

    def bfs(self, start):
        visited = set()
        queue = deque([(start, [])])
        nodes = 0
        while queue:
            state, path = queue.popleft()
            if state == self.goal:
                return path, nodes
            visited.add(state)
            for neighbor, move in self.get_neighbors(state):
                if neighbor not in visited:
                    queue.append((neighbor, path + [move]))
            nodes += 1
        return None, nodes

    def a_star(self, start, heuristic='manhattan'):
        visited = set()
        h_func = self.manhattan if heuristic == 'manhattan' else self.misplaced
        heap = [(h_func(start), 0, start, [])]
        nodes = 0
        while heap:
            f, g, state, path = heapq.heappop(heap)
            if state == self.goal:
                return path, nodes
            if state in visited:
                continue
            visited.add(state)
            for neighbor, move in self.get_neighbors(state):
                if neighbor not in visited:
                    h = h_func(neighbor)
                    heapq.heappush(heap, (g + 1 + h, g + 1, neighbor, path + [move]))
            nodes += 1
        return None, nodes

    def run_tests(self, steps_list=[5, 10, 20, 40, 80], instances=3):
        for steps in steps_list:
            for i in range(instances):
                start, _ = self.random_walk(steps)
                print(f"\n{self.size}x{self.size} Puzzle - {steps} steps, instance {i+1}:")
                print("Start:", start)
                path_bfs, nodes_bfs = self.bfs(start)
                path_astar_m = self.a_star(start, 'manhattan')
                path_astar_o = self.a_star(start, 'misplaced')
                print("  BFS: Length =", len(path_bfs), ", Nodes =", nodes_bfs)
                print("  A* Manhattan: Length =", len(path_astar_m[0]), ", Nodes =", path_astar_m[1])
                print("  A* Misplaced: Length =", len(path_astar_o[0]), ", Nodes =", path_astar_o[1])

if __name__ == "__main__":
    print("--- Testing 3x3 ---")
    puzzle3 = SlidingPuzzle(3)
    puzzle3.run_tests()

    print("\n--- Testing 4x4 ---")
    puzzle4 = SlidingPuzzle(4)
    puzzle4.run_tests()

--- Testing 3x3 ---

3x3 Puzzle - 5 steps, instance 1:
Start: (1, 3, 5, 4, 2, 0, 7, 8, 6)
  BFS: Length = 5 , Nodes = 38
  A* Manhattan: Length = 5 , Nodes = 6
  A* Misplaced: Length = 5 , Nodes = 7

3x3 Puzzle - 5 steps, instance 2:
Start: (2, 0, 3, 1, 5, 6, 4, 7, 8)
  BFS: Length = 5 , Nodes = 46
  A* Manhattan: Length = 5 , Nodes = 5
  A* Misplaced: Length = 5 , Nodes = 5

3x3 Puzzle - 5 steps, instance 3:
Start: (4, 1, 2, 0, 5, 3, 7, 8, 6)
  BFS: Length = 5 , Nodes = 39
  A* Manhattan: Length = 5 , Nodes = 5
  A* Misplaced: Length = 5 , Nodes = 5

3x3 Puzzle - 10 steps, instance 1:
Start: (2, 8, 0, 1, 5, 3, 4, 7, 6)
  BFS: Length = 10 , Nodes = 512
  A* Manhattan: Length = 10 , Nodes = 17
  A* Misplaced: Length = 10 , Nodes = 29

3x3 Puzzle - 10 steps, instance 2:
Start: (4, 1, 2, 5, 0, 8, 7, 6, 3)
  BFS: Length = 10 , Nodes = 969
  A* Manhattan: Length = 10 , Nodes = 15
  A* Misplaced: Length = 10 , Nodes = 43

3x3 Puzzle - 10 steps, instance 3:
Start: (1, 3, 6, 4, 5, 2, 0, 7, 8)


In the above problem solving with search project, I coded a simple sliding puzzle code to support both the 3x3 and 4x4 puzzles and tested the performance of different saerch algorithms firstly I used  (bfs) and then A* with MAanhattan Distance heurstic.

Also for each puzzle I generated 15 random problems usign random walks of different lengths (5,10,20,40,80), and recorded the solution length and number of nodes expanded.

After observing the results, its clean that the BFS works best for smaller problems but quickly becomes not sufficient as the problem pize or scarmble depth increses. In some cases, espically with the 4x4 puzzle, BFS expanded so many nodes before finding the right solution, which makes it not practical. Meanwhile, A* search worked like a star and was more effcient, mostly when using Manhattan distance heuristic.

**Final Conclusion:** This solving with search project shows us that why heuristic based search is important in AI. As problems get more complex, having a good heuristic is the key to keeping search fast and memory efficient.